In [1]:
import torch
text=('python models learn from examples. students inspect errors and improve models. ')*30
chars=sorted(set(text));to_id={c:i for i,c in enumerate(chars)};to_char={i:c for c,i in to_id.items()}
encoded=torch.tensor([to_id[c] for c in text],dtype=torch.long)
print(len(text),len(chars),chars)

2370 21 [' ', '.', 'a', 'c', 'd', 'e', 'f', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'x', 'y']


When a character isn’t in the vocabulary, it has no ID, so it can’t be turned into a number. The model would not be able to see that character and can't predict it.

In [2]:
n=len(encoded);train_ids=encoded[:int(.7*n)];val_ids=encoded[int(.7*n):int(.85*n)];test_ids=encoded[int(.85*n):]
window=20
def pairs(ids):
    X=torch.stack([ids[i:i+window] for i in range(len(ids)-window)])
    y=ids[window:]
    return X,y
X_train,y_train=pairs(train_ids);X_val,y_val=pairs(val_ids);X_test,y_test=pairs(test_ids)
print(X_train.shape,X_val.shape,X_test.shape)

torch.Size([1639, 20]) torch.Size([335, 20]) torch.Size([336, 20])


The test text is the same sentence the model already had during training. It means that the model can score well just by remembering it, even if it couldn’t handle new text.

In [3]:
from torch import nn
class CharGRU(nn.Module):
    def __init__(self,vocab):
        super().__init__();self.embed=nn.Embedding(vocab,8);self.gru=nn.GRU(8,24,batch_first=True);self.head=nn.Linear(24,vocab)
    def forward(self,x):
        out,_=self.gru(self.embed(x));return self.head(out[:,-1,:])
torch.manual_seed(42);model=CharGRU(len(chars));print(model(X_train[:4]).shape)

torch.Size([4, 21])


Since the next character could be any of the 21 in the vocabulary, the model gives every one of them a score. Whichever character gets the highest score, that's the one it will predict.

In [4]:
from torch.utils.data import TensorDataset,DataLoader
from collections import Counter
majority=Counter(y_train.tolist()).most_common(1)[0][0]
print('majority validation accuracy:',(y_val==majority).float().mean().item())
loader=DataLoader(TensorDataset(X_train,y_train),batch_size=64,shuffle=True)
opt=torch.optim.Adam(model.parameters(),lr=.01);loss_fn=nn.CrossEntropyLoss()
for epoch in range(15):
    model.train()
    for xb,yb in loader:
        opt.zero_grad();loss=loss_fn(model(xb),yb);loss.backward();opt.step()
model.eval()
with torch.inference_mode():val_pred=model(X_val).argmax(1)
print('validation accuracy:',(val_pred==y_val).float().mean().item())

majority validation accuracy: 0.13731342554092407
validation accuracy: 1.0


A low loss just means that the model is good at guessing the next character in text like what it had trained on. It doesn't understand meaning, so when it writes a longer piece one character at a time, the result can still come out awkward and repetitive.

In [5]:
prompt=text[:window]
input_ids=torch.tensor([[to_id[c] for c in prompt]])
with torch.inference_mode():next_id=model(input_ids).argmax(1).item()
print(repr(prompt), '->',repr(to_char[next_id]))

'python models learn ' -> 'f'


If the model learned from a lot more text with a lot of different sentences, then it would learn real language patterns instead of just memorizing one sentence. Then it would make good predictions on new text as well.

In [6]:
with torch.inference_mode():test_pred=model(X_test).argmax(1)
print('final test accuracy:',(test_pred==y_test).float().mean().item())

final test accuracy: 1.0


Next-character prediction is only able to guess which letter comes next based on patterns the model has seen before. Judging meaning means understanding what the words and sentences are actually saying, this model doesn’t do that.

### Debrief

The baseline model always guessed the most common character. It reached about 13.7% accuracy on the validation data. The GRU model did much better, reaching 100% accuracy on both the validation data and the final test. However, this result is limited because the text is the same sentence repeated 30 times, so the test data has the same patterns the model already saw during its training. This means the model scored perfectly by memorizing the sentence, not because it can handle new text. A good next controlled experiment would be to train the model on a larger text with many different sentences while keeping everything else the same and then compare its validation accuracy to the baseline again.